# Multi Tool Use

Real agents need multiple tools and must choose the right one.
Sometimes they even call multiple tools in a single turn.

In this notebook:
- Give LLM 3+ tools
- LLM picks the right one automatically
- Handle cases where LLM calls multiple tools at once
- Build a reusable agent loop

# Setup

In [4]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

# All tools + functions

In [5]:
def calculate(operation: str, a: float, b: float):
    ops = {"add": a+b, "subtract": a-b, "multiply": a*b, "divide": a/b if b!=0 else "Error"}
    return str(ops.get(operation, "Unknown operation"))

In [6]:
def get_weather(city: str, unit: str = "celcius"):
    return f"{city}: 30°C, Sunny"

In [7]:
def search_web(query: str):
    return f"Top results for '{query}':"

In [8]:
def read_file(filename: str):
    return f"Contents of {filename}:"

In [9]:
available_tools = {
    "calculate": calculate,
    "get_weather": get_weather,
    "search_web": search_web,
    "read_file": read_file,
}

In [10]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform math. Use for any calculations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
                    "a": {"type": "number"},
                    "b": {"type": "number"}
                },
                "required": ["operation", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather for the city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "unit": {"type": "string", "enum": ["celcius", "fahrenheit"]}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search web for current information and recent events",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read contents of a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string"}
                },
                "required": ["filename"]
            }
        }
    }
]

# Reuseable Agent Loop

In [19]:
def execute_tool_calls(tool_calls: list):
    results = []
    for tool_call in tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print(f"Calling {name} ({args})")

        if name in available_tools:
            result = available_tools[name](**args)
        else:
            result = f"Error: tool {name} not found"

        results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
    return results

### Full agent loop:

   1. LLM thinks

   2. If tool call → execute → feed back → repeat

   3. If direct answer → return

In [22]:
def agent_loop(user_message: str, max_iterations: int = 5):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
        {"role": "user", "content": user_message}
    ]

    for i in range(max_iterations):
        print(f"Iteration: {i+1}")

        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=tools,
            temperature=0
        )
        message = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "stop":
            print("LLM gave final answer")
            return message.content

        if finish_reason == "tool_calls":
            print(f"LLM calling {len(message.tool_calls)} tool(s)")

            messages.append({"role": "assistant", "tool_calls": message.tool_calls})

            tool_results = execute_tool_calls(message.tool_calls)

            messages.extend(tool_results)

    return "Max iterations reached"

# Test with different queries

In [23]:
print(agent_loop("What is 156 multiplied by 48?"))

print(agent_loop("What's the weather like in Karachi?"))

print(agent_loop("What is the speed of light?"))

Iteration: 1
LLM calling 1 tool(s)
Calling calculate ({'a': 156, 'b': 48, 'operation': 'multiply'})
Iteration: 2
LLM gave final answer
156 multiplied by 48 equals **7,488**.
Iteration: 1
LLM calling 1 tool(s)
Calling get_weather ({'city': 'Karachi', 'unit': 'fahrenheit'})
Iteration: 2
LLM gave final answer
Karachi is currently sunny with a temperature of **30 °C** (about **86 °F**).
Iteration: 1
LLM gave final answer
The speed of light in a vacuum is a fundamental constant of nature, denoted by \(c\). Its value is:

\[
c \;=\; 299{,}792{,}458 \text{ meters per second (m/s)}
\]

This is an exact value defined by the International System of Units (SI). In other units, it’s about 186,282 miles per second or roughly 1,079,252,848.8 kilometers per hour.
